In [0]:
from pyspark.sql.functions import *

catalog = "bike_data"
bronze_schema = "bronze"
silver_schema = "silver"

print("=" * 80)
print("SILVER TRANSFORMATION: crm_cust_info")
print("=" * 80)

# Section 1: Read Bronze table
print("\nSection 1: Reading Bronze table")
df = spark.table(f"{catalog}.{bronze_schema}.crm_cust_info")
initial_count = df.count()
print(f"Initial rows: {initial_count}")

# Section 2: Transform data
print("\nSection 2: Data transformation")

# 2.1 Remove duplicates
print("  2.1 Removing duplicates by cst_id")
df = df.dropDuplicates(subset=["cst_id"])
print(f"  Rows after dedup: {df.count()}")

# 2.2 Clean string columns
print("  2.2 Cleaning string columns")
df = df.withColumn("cst_firstname", trim(upper(col("cst_firstname"))))
df = df.withColumn("cst_lastname", trim(upper(col("cst_lastname"))))
df = df.withColumn("cst_marital_status", trim(upper(col("cst_marital_status"))))
df = df.withColumn("cst_gndr", trim(upper(col("cst_gndr"))))

# 2.3 Fix date columns
print("  2.3 Fixing date columns")
df = df.withColumn("cst_create_date",
    coalesce(
        to_date(col("cst_create_date"), "yyyy-MM-dd"),
        to_date(col("cst_create_date"), "MM/dd/yyyy"),
        to_date(col("cst_create_date"), "dd-MM-yyyy")
    )
)

# 2.4 Remove invalid rows
print("  2.4 Removing invalid rows")
df = df.filter(col("cst_id").isNotNull())

print(f"  Rows after transformations: {df.count()}")

# Section 3: Sanity checks
print("\nSection 3: Sanity checks")
null_check = df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns])
display(null_check)
display(df.limit(3))

# Section 4: Write to Silver
print("\nSection 4: Writing to Silver table")
silver_table = f"{catalog}.{silver_schema}.customers"
df.write.mode("overwrite").format("delta").saveAsTable(silver_table)

final_count = df.count()
print(f"Written to: {silver_table}")
print(f"Final rows: {final_count}")
print(f"Removed: {initial_count - final_count} rows")